# Predial Vision MX — Phase 1: TTA + Geometric Filter + Threshold Optimization
v12 retrain + 6-augmentation TTA + shapely geometric filtering
Target: F1 0.49 → 0.55-0.58

In [ ]:
import os, sys, gc, time, json, gzip, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from urllib.request import urlretrieve
import subprocess

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.models.segmentation import deeplabv3_mobilenet_v3_large

import rasterio
from rasterio.transform import from_bounds
from rasterio.features import rasterize, shapes
from rasterio.crs import CRS
import fiona
import shapely
from shapely.geometry import shape, mapping, box
from shapely.ops import unary_union
import geopandas as gpd
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
print('Libraries loaded successfully')

## 1. GPU Detection

In [ ]:
def get_gpu_info():
    if not torch.cuda.is_available():
        return None, None, None
    props = torch.cuda.get_device_properties(0)
    cc = props.major * 10 + props.minor  # compute capability as int
    name = props.name
    mem_gb = props.total_memory / 1e9
    return cc, name, mem_gb

cc, gpu_name, gpu_mem = get_gpu_info()

if cc is None:
    print('WARNING: No GPU detected — training will be very slow on CPU')
    DEVICE = 'cpu'
    SKIP_TRAINING = False
elif cc < 75:  # sm_60 = P100, sm_70 = V100-limited
    print(f'WARNING: GPU {gpu_name} has compute capability sm_{cc} (< sm_75)')
    print('P100 (sm_60) may cause issues with some ops. Proceeding cautiously.')
    DEVICE = 'cuda'
    SKIP_TRAINING = False
else:
    print(f'GPU: {gpu_name} | CC: sm_{cc} | VRAM: {gpu_mem:.1f} GB')
    DEVICE = 'cuda'
    SKIP_TRAINING = False

print(f'Device: {DEVICE}')

## 2. Configuration

In [ ]:
# Paths
WORKING_DIR = Path('/kaggle/working')
INPUT_DIR   = Path('/kaggle/input')
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# MS Buildings download
MS_BUILDINGS_URL = 'https://github.com/MarxCha/predial-vision-mx/releases/download/v0.2.0-data/ms_buildings_nr.geojson.gz'
MS_BUILDINGS_PATH = WORKING_DIR / 'ms_buildings_nr.geojson.gz'
MS_BUILDINGS_GEOJSON = WORKING_DIR / 'ms_buildings_nr.geojson'

# Model
CHIP_SIZE   = 256
STRIDE_TRAIN = 128
STRIDE_VAL   = 256
STRIDE_INFER = 128
BATCH_SIZE   = 16
NUM_EPOCHS   = 50
LEARNING_RATE = 1e-3
POS_WEIGHT    = 5.0
VAL_SPLIT     = 0.30
RANDOM_SEED   = 42

# Inference
THRESHOLDS = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
PRED_THRESHOLD = 0.35  # default, will be updated after sweep

# Geometric filter
MIN_AREA_M2  = 20
MIN_RECT     = 0.45
MIN_COMPACT  = 0.1

# Labels
BUILDINGS_COUNT = 0
LABEL_SOURCE    = 'MS Buildings (GitHub Release v0.2.0-data)'

# Output
MODEL_PATH    = WORKING_DIR / 'best_model.pth'
EVAL_PATH     = WORKING_DIR / 'eval.json'
GEOJSON_PATH  = WORKING_DIR / 'phase1_buildings.geojson'
PROB_MAP_PATH = WORKING_DIR / 'phase1_prob_map.npz'
VIZ_PATH      = WORKING_DIR / 'phase1_visualization.png'
CURVES_PATH   = WORKING_DIR / 'training_curves.png'

print('Configuration set')

## 3. Download MS Buildings Labels

In [ ]:
def download_ms_buildings():
    if MS_BUILDINGS_GEOJSON.exists():
        print(f'MS Buildings GeoJSON already exists: {MS_BUILDINGS_GEOJSON}')
        return True
    if not MS_BUILDINGS_PATH.exists():
        print(f'Downloading MS Buildings from: {MS_BUILDINGS_URL}')
        try:
            urlretrieve(MS_BUILDINGS_URL, str(MS_BUILDINGS_PATH))
            print(f'Downloaded: {MS_BUILDINGS_PATH.stat().st_size / 1e6:.1f} MB')
        except Exception as e:
            print(f'Download failed: {e}')
            return False
    print('Decompressing...')
    with gzip.open(str(MS_BUILDINGS_PATH), 'rb') as f_in:
        with open(str(MS_BUILDINGS_GEOJSON), 'wb') as f_out:
            f_out.write(f_in.read())
    print(f'Decompressed: {MS_BUILDINGS_GEOJSON.stat().st_size / 1e6:.1f} MB')
    return True

ms_ok = download_ms_buildings()
print(f'MS Buildings ready: {ms_ok}')

## 4. Load Imagery

In [ ]:
def find_imagery():
    tif_files = list(INPUT_DIR.rglob('*.tif')) + list(INPUT_DIR.rglob('*.tiff'))
    tif_files = [f for f in tif_files if f.stat().st_size > 1e6]  # > 1MB
    print(f'Found {len(tif_files)} GeoTIFF file(s) in /kaggle/input')
    for f in tif_files:
        print(f'  {f}  ({f.stat().st_size/1e6:.1f} MB)')
    return tif_files

tif_files = find_imagery()

if not tif_files:
    print('No imagery found in /kaggle/input — generating synthetic tile for testing...')
    # Fallback: create a small synthetic tile in Nuevo Reynosa bbox
    SYNTH_PATH = WORKING_DIR / 'synthetic_test.tif'
    # Nuevo Reynosa approximate bbox
    west, south, east, north = -98.35, 26.00, -98.25, 26.08
    width, height = 512, 512
    transform = from_bounds(west, south, east, north, width, height)
    data = np.random.randint(50, 200, (3, height, width), dtype=np.uint8)
    with rasterio.open(
        str(SYNTH_PATH), 'w',
        driver='GTiff', height=height, width=width, count=3,
        dtype='uint8', crs=CRS.from_epsg(4326), transform=transform
    ) as dst:
        dst.write(data)
    tif_files = [SYNTH_PATH]
    print(f'Synthetic tile created: {SYNTH_PATH}')

# Use first (or largest) tif
IMAGE_PATH = max(tif_files, key=lambda f: f.stat().st_size)
print(f'\nUsing image: {IMAGE_PATH}')

with rasterio.open(str(IMAGE_PATH)) as src:
    IMG_CRS       = src.crs
    IMG_TRANSFORM = src.transform
    IMG_HEIGHT    = src.height
    IMG_WIDTH     = src.width
    IMG_BANDS     = src.count
    IMG_BOUNDS    = src.bounds
    print(f'  Size: {IMG_WIDTH} x {IMG_HEIGHT} px | Bands: {IMG_BANDS}')
    print(f'  CRS: {IMG_CRS}')
    print(f'  Bounds: {IMG_BOUNDS}')

## 5. Rasterize MS Buildings to Mask

In [ ]:
def rasterize_buildings(geojson_path, image_path, chunk_size=2048):
    """Chunked, memory-efficient rasterization of MS Buildings to match image grid."""
    global BUILDINGS_COUNT

    with rasterio.open(str(image_path)) as src:
        crs       = src.crs
        transform = src.transform
        height    = src.height
        width     = src.width
        bounds    = src.bounds

    print(f'Loading MS Buildings from {geojson_path}...')
    gdf = gpd.read_file(str(geojson_path))
    BUILDINGS_COUNT = len(gdf)
    print(f'  Total buildings in file: {BUILDINGS_COUNT:,}')

    # Reproject to image CRS if needed
    if gdf.crs is None:
        gdf = gdf.set_crs('EPSG:4326')
    if gdf.crs != crs:
        print(f'  Reprojecting from {gdf.crs} to {crs}...')
        gdf = gdf.to_crs(crs)

    # Clip to image bounds
    img_bbox = box(bounds.left, bounds.bottom, bounds.right, bounds.top)
    gdf = gdf[gdf.geometry.intersects(img_bbox)].copy()
    BUILDINGS_COUNT = len(gdf)
    print(f'  Buildings within image bounds: {BUILDINGS_COUNT:,}')

    if BUILDINGS_COUNT == 0:
        print('  WARNING: No buildings overlap with image — using zero mask')
        return np.zeros((height, width), dtype=np.uint8)

    # Build shapes list
    shapes_list = [(geom, 1) for geom in gdf.geometry if geom is not None and geom.is_valid]

    # Chunked rasterization
    mask = np.zeros((height, width), dtype=np.uint8)
    n_chunks_y = math.ceil(height / chunk_size)
    n_chunks_x = math.ceil(width / chunk_size)
    total_chunks = n_chunks_y * n_chunks_x
    done = 0

    for cy in range(n_chunks_y):
        row_off = cy * chunk_size
        row_end = min(row_off + chunk_size, height)
        for cx in range(n_chunks_x):
            col_off = cx * chunk_size
            col_end = min(col_off + chunk_size, width)
            ch = row_end - row_off
            cw = col_end - col_off

            chunk_transform = rasterio.transform.from_origin(
                transform.c + col_off * transform.a,
                transform.f + row_off * transform.e,
                transform.a,
                -transform.e
            )

            chunk_mask = rasterize(
                shapes_list,
                out_shape=(ch, cw),
                transform=chunk_transform,
                fill=0,
                dtype=np.uint8,
                all_touched=True
            )
            mask[row_off:row_end, col_off:col_end] = chunk_mask
            done += 1
            if done % max(1, total_chunks // 10) == 0:
                print(f'  Rasterized chunk {done}/{total_chunks} ({100*done/total_chunks:.0f}%)')

    coverage = mask.mean() * 100
    print(f'  Mask coverage: {coverage:.2f}% positive pixels')
    return mask

if ms_ok and MS_BUILDINGS_GEOJSON.exists():
    MASK = rasterize_buildings(MS_BUILDINGS_GEOJSON, IMAGE_PATH)
else:
    print('MS Buildings not available — creating zero mask')
    MASK = np.zeros((IMG_HEIGHT, IMG_WIDTH), dtype=np.uint8)
    BUILDINGS_COUNT = 0

print(f'\nMask shape: {MASK.shape} | dtype: {MASK.dtype}')
print(f'Buildings used as labels: {BUILDINGS_COUNT:,}')

## 6. Dataset and Augmentations

In [ ]:
def read_image_full(image_path):
    """Read full image as HxWxC uint8 numpy array."""
    with rasterio.open(str(image_path)) as src:
        bands = []
        for b in range(1, min(src.count + 1, 4)):  # max 3 bands (RGB)
            band = src.read(b).astype(np.float32)
            # Normalize to 0-255
            p2, p98 = np.percentile(band[band > 0], [2, 98]) if band.any() else (0, 255)
            band = np.clip((band - p2) / max(p98 - p2, 1e-6) * 255, 0, 255)
            bands.append(band.astype(np.uint8))
        if len(bands) == 1:
            bands = bands * 3  # replicate grayscale to 3 channels
        elif len(bands) == 2:
            bands.append(bands[0])
        img = np.stack(bands[:3], axis=2)  # HxWx3
    return img

print('Reading full image...')
IMG_ARRAY = read_image_full(IMAGE_PATH)
print(f'Image array shape: {IMG_ARRAY.shape} | dtype: {IMG_ARRAY.dtype}')
print(f'Mask shape: {MASK.shape}')

In [ ]:
def extract_chips(image, mask, chip_size, stride):
    """Extract overlapping chips from image and mask."""
    H, W = image.shape[:2]
    chips_img, chips_msk = [], []
    for y in range(0, H - chip_size + 1, stride):
        for x in range(0, W - chip_size + 1, stride):
            chip_img = image[y:y+chip_size, x:x+chip_size]
            chip_msk = mask[y:y+chip_size, x:x+chip_size]
            chips_img.append(chip_img)
            chips_msk.append(chip_msk)
    return chips_img, chips_msk

print('Extracting training chips...')
all_imgs, all_msks = extract_chips(IMG_ARRAY, MASK, CHIP_SIZE, STRIDE_TRAIN)
print(f'Total chips (stride={STRIDE_TRAIN}): {len(all_imgs)}')

# Train/val split
indices = list(range(len(all_imgs)))
train_idx, val_idx = train_test_split(indices, test_size=VAL_SPLIT, random_state=RANDOM_SEED)
print(f'Train: {len(train_idx)} | Val: {len(val_idx)}')

In [ ]:
class BuildingDataset(Dataset):
    def __init__(self, images, masks, transform=None):
        self.images    = images
        self.masks     = masks
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img  = self.images[idx].copy()
        mask = self.masks[idx].copy().astype(np.float32)
        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img  = augmented['image']
            mask = augmented['mask']
        return img, mask.unsqueeze(0) if isinstance(mask, torch.Tensor) else torch.from_numpy(mask).unsqueeze(0)

train_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.HueSaturationValue(p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_aug = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

train_imgs = [all_imgs[i] for i in train_idx]
train_msks = [all_msks[i] for i in train_idx]
val_imgs   = [all_imgs[i] for i in val_idx]
val_msks   = [all_msks[i] for i in val_idx]

train_ds = BuildingDataset(train_imgs, train_msks, train_aug)
val_ds   = BuildingDataset(val_imgs,   val_msks,   val_aug)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

## 7. Model, Loss, Optimizer

In [ ]:
class DiceBCELoss(nn.Module):
    def __init__(self, pos_weight=5.0):
        super().__init__()
        self.pos_weight = torch.tensor([pos_weight])
        self.bce_loss   = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]))

    def forward(self, logits, targets):
        # BCEWithLogitsLoss
        self.bce_loss.pos_weight = self.bce_loss.pos_weight.to(logits.device)
        bce = self.bce_loss(logits, targets)

        # Dice
        probs   = torch.sigmoid(logits)
        smooth  = 1.0
        inter   = (probs * targets).sum()
        dice    = 1.0 - (2.0 * inter + smooth) / (probs.sum() + targets.sum() + smooth)

        return bce + dice

def build_model():
    model = deeplabv3_mobilenet_v3_large(weights='DEFAULT')
    # Replace classifier head: 256 -> 1 output channel
    model.classifier[4] = nn.Conv2d(256, 1, kernel_size=1)
    if hasattr(model, 'aux_classifier') and model.aux_classifier is not None:
        model.aux_classifier[4] = nn.Conv2d(10, 1, kernel_size=1)
    return model

print('Building model...')
model = build_model().to(DEVICE)
criterion = DiceBCELoss(pos_weight=POS_WEIGHT)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5
)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')
print('Model ready')

## 8. Training Loop

In [ ]:
def compute_iou(pred, target, threshold=0.5):
    pred_bin  = (pred > threshold).float()
    inter     = (pred_bin * target).sum().item()
    union     = ((pred_bin + target) > 0).float().sum().item()
    return inter / (union + 1e-7)

def compute_f1(pred, target, threshold=0.5):
    pred_bin = (pred > threshold).float()
    tp = (pred_bin * target).sum().item()
    fp = (pred_bin * (1 - target)).sum().item()
    fn = ((1 - pred_bin) * target).sum().item()
    prec = tp / (tp + fp + 1e-7)
    rec  = tp / (tp + fn + 1e-7)
    f1   = 2 * prec * rec / (prec + rec + 1e-7)
    return f1, prec, rec

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device).float()
        optimizer.zero_grad()
        out  = model(imgs)['out']
        out  = nn.functional.interpolate(out, size=masks.shape[-2:], mode='bilinear', align_corners=False)
        loss = criterion(out, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def val_epoch(model, loader, criterion, device, threshold=0.35):
    model.eval()
    total_loss, total_f1 = 0, 0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device).float()
        out   = model(imgs)['out']
        out   = nn.functional.interpolate(out, size=masks.shape[-2:], mode='bilinear', align_corners=False)
        loss  = criterion(out, masks)
        probs = torch.sigmoid(out)
        f1, _, _ = compute_f1(probs, masks)
        total_loss += loss.item()
        total_f1   += f1
    return total_loss / len(loader), total_f1 / len(loader)

print('Training functions defined')

In [ ]:
train_losses, val_losses, val_f1s = [], [], []
best_val_f1 = 0.0
best_epoch  = 0

print(f'Starting training: {NUM_EPOCHS} epochs | device: {DEVICE}')
print('-' * 60)

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    tr_loss = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
    vl_loss, vl_f1 = val_epoch(model, val_loader, criterion, DEVICE)

    train_losses.append(tr_loss)
    val_losses.append(vl_loss)
    val_f1s.append(vl_f1)

    scheduler.step(vl_f1)

    if vl_f1 > best_val_f1:
        best_val_f1 = vl_f1
        best_epoch  = epoch
        torch.save(model.state_dict(), str(MODEL_PATH))

    elapsed = time.time() - t0
    lr_now  = optimizer.param_groups[0]['lr']
    if epoch % 5 == 0 or epoch == 1:
        marker = ' <-- best' if epoch == best_epoch else ''
        print(f'Epoch {epoch:3d}/{NUM_EPOCHS} | '
              f'TrLoss: {tr_loss:.4f} | VlLoss: {vl_loss:.4f} | '
              f'VlF1: {vl_f1:.4f} | LR: {lr_now:.2e} | {elapsed:.1f}s{marker}')

print('-' * 60)
print(f'Best val F1: {best_val_f1:.4f} @ epoch {best_epoch}')
print(f'Model saved: {MODEL_PATH}')

In [ ]:
# Save training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, label='Train Loss', color='steelblue')
axes[0].plot(val_losses,   label='Val Loss',   color='coral')
axes[0].axvline(best_epoch - 1, color='green', linestyle='--', alpha=0.7, label=f'Best epoch {best_epoch}')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(val_f1s, label='Val F1', color='mediumseagreen')
axes[1].axvline(best_epoch - 1, color='green', linestyle='--', alpha=0.7, label=f'Best epoch {best_epoch}')
axes[1].axhline(best_val_f1, color='orange', linestyle=':', alpha=0.7, label=f'Best F1={best_val_f1:.4f}')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('F1 Score')
axes[1].set_title('Validation F1 Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Phase 1 Training — Predial Vision MX', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(CURVES_PATH), dpi=150, bbox_inches='tight')
plt.show()
print(f'Training curves saved: {CURVES_PATH}')

## 9. TTA Inference

In [ ]:
@torch.no_grad()
def tta_predict_chip(model, chip, device):
    """chip: (1, C, H, W) tensor on device. Returns averaged probability (1, 1, H, W)"""
    preds = []

    # Original
    out = torch.sigmoid(model(chip)['out'])
    preds.append(out)

    # Horizontal flip
    flipped = torch.flip(chip, dims=[3])
    out = torch.sigmoid(model(flipped)['out'])
    preds.append(torch.flip(out, dims=[3]))

    # Vertical flip
    flipped = torch.flip(chip, dims=[2])
    out = torch.sigmoid(model(flipped)['out'])
    preds.append(torch.flip(out, dims=[2]))

    # Rotate 90
    rotated = torch.rot90(chip, k=1, dims=[2, 3])
    out = torch.sigmoid(model(rotated)['out'])
    preds.append(torch.rot90(out, k=3, dims=[2, 3]))

    # Rotate 180
    rotated = torch.rot90(chip, k=2, dims=[2, 3])
    out = torch.sigmoid(model(rotated)['out'])
    preds.append(torch.rot90(out, k=2, dims=[2, 3]))

    # Rotate 270
    rotated = torch.rot90(chip, k=3, dims=[2, 3])
    out = torch.sigmoid(model(rotated)['out'])
    preds.append(torch.rot90(out, k=1, dims=[2, 3]))

    return torch.stack(preds).mean(dim=0)

print('TTA function defined — 6 augmentations per chip')

In [ ]:
# Load best model
print(f'Loading best model from {MODEL_PATH}...')
model.load_state_dict(torch.load(str(MODEL_PATH), map_location=DEVICE))
model.eval()
print('Best model loaded')

# Normalization transform (no augmentation for inference)
infer_norm = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

H, W = IMG_ARRAY.shape[:2]
pred_mask  = np.zeros((H, W), dtype=np.float32)
count_mask = np.zeros((H, W), dtype=np.float32)

# Sliding window with TTA
ys = list(range(0, H - CHIP_SIZE + 1, STRIDE_INFER))
xs = list(range(0, W - CHIP_SIZE + 1, STRIDE_INFER))
total_chips = len(ys) * len(xs)
print(f'TTA inference: {total_chips} chips (stride={STRIDE_INFER}, 6 passes each)...')

done_chips = 0
log_every  = max(1, total_chips // 20)

for y in ys:
    for x in xs:
        chip = IMG_ARRAY[y:y+CHIP_SIZE, x:x+CHIP_SIZE]

        # Normalize
        aug = infer_norm(image=chip)
        tensor = aug['image'].unsqueeze(0).to(DEVICE)  # (1, C, H, W)

        # TTA: 6 forward passes
        prob = tta_predict_chip(model, tensor, DEVICE)  # (1, 1, H, W)
        prob_np = prob.squeeze().cpu().numpy()           # (H, W)

        # Resize if needed (bilinear interpolation handles non-square)
        if prob_np.shape != (CHIP_SIZE, CHIP_SIZE):
            from PIL import Image as PILImage
            prob_pil = PILImage.fromarray(prob_np)
            prob_np  = np.array(prob_pil.resize((CHIP_SIZE, CHIP_SIZE), PILImage.BILINEAR))

        pred_mask[y:y+CHIP_SIZE, x:x+CHIP_SIZE]  += prob_np
        count_mask[y:y+CHIP_SIZE, x:x+CHIP_SIZE] += 1.0

        done_chips += 1
        if done_chips % log_every == 0:
            pct = 100 * done_chips / total_chips
            print(f'  {done_chips}/{total_chips} chips ({pct:.0f}%)')

# Average overlapping predictions
valid = count_mask > 0
pred_mask[valid] /= count_mask[valid]

print(f'\nTTA inference complete')
print(f'Pred mask range: [{pred_mask.min():.3f}, {pred_mask.max():.3f}]')

# Save probability map
np.savez_compressed(str(PROB_MAP_PATH), prob=pred_mask)
print(f'Probability map saved: {PROB_MAP_PATH}')

## 10. Threshold Sweep

In [ ]:
def evaluate_at_threshold(pred, target, threshold):
    pred_bin = (pred > threshold).astype(np.float32)
    tp = (pred_bin * target).sum()
    fp = (pred_bin * (1 - target)).sum()
    fn = ((1 - pred_bin) * target).sum()
    tn = ((1 - pred_bin) * (1 - target)).sum()
    prec  = tp / (tp + fp + 1e-7)
    rec   = tp / (tp + fn + 1e-7)
    f1    = 2 * prec * rec / (prec + rec + 1e-7)
    inter = tp
    union = tp + fp + fn
    iou   = inter / (union + 1e-7)
    return float(prec), float(rec), float(f1), float(iou)

mask_float = MASK.astype(np.float32)
sweep_results = {}

print('Threshold sweep vs MS Buildings mask:')
print(f'{"Threshold":>10} | {"Precision":>10} | {"Recall":>8} | {"F1":>8}')
print('-' * 46)

best_f1   = 0.0
best_thresh = 0.35

for t in THRESHOLDS:
    p, r, f1, iou = evaluate_at_threshold(pred_mask, mask_float, t)
    sweep_results[t] = {'precision': p, 'recall': r, 'f1': f1, 'iou': iou}
    marker = ' <-- best' if f1 > best_f1 else ''
    if f1 > best_f1:
        best_f1     = f1
        best_thresh = t
    print(f'{t:>10.2f} | {p:>10.4f} | {r:>8.4f} | {f1:>8.4f}{marker}')

PRED_THRESHOLD = best_thresh
print(f'\nBest threshold: {PRED_THRESHOLD:.2f} (F1={best_f1:.4f})')

## 11. Vectorize Predictions

In [ ]:
def vectorize_mask(binary_mask, transform, crs):
    """Convert binary mask to list of shapely polygons."""
    polys = []
    for geom, val in shapes(binary_mask.astype(np.uint8), transform=transform):
        if val == 1:
            poly = shape(geom)
            if poly.is_valid and not poly.is_empty:
                polys.append(poly)
    return polys

binary_pred = (pred_mask > PRED_THRESHOLD).astype(np.uint8)
print(f'Vectorizing binary mask (threshold={PRED_THRESHOLD})...')

raw_polys = vectorize_mask(binary_pred, IMG_TRANSFORM, IMG_CRS)
print(f'Raw polygons before geometric filter: {len(raw_polys):,}')

## 12. Geometric Filtering

In [ ]:
def rectangularity(poly):
    mbr = poly.minimum_rotated_rectangle
    return poly.area / mbr.area if mbr.area > 0 else 0.0

def compactness(poly):
    p = poly.length
    return (4 * np.pi * poly.area / (p * p)) if p > 0 else 0.0

def filter_building(poly, min_area_m2=20, min_rect=0.45, min_compact=0.1):
    area_m2 = poly.area * (111000 ** 2)  # degrees^2 -> approx m^2
    if area_m2 < min_area_m2:
        return False
    if rectangularity(poly) < min_rect:
        return False
    if compactness(poly) < min_compact:
        return False
    return True

before_count = len(raw_polys)
filtered_polys = [p for p in raw_polys if filter_building(p, MIN_AREA_M2, MIN_RECT, MIN_COMPACT)]
after_count  = len(filtered_polys)
removed      = before_count - after_count

print(f'Before geometric filter: {before_count:,} polygons')
print(f'After geometric filter:  {after_count:,} polygons (removed {removed:,})')

## 13. Final Evaluation

In [ ]:
# Re-rasterize filtered polygons for evaluation
print('Rasterizing filtered predictions for final evaluation...')

if filtered_polys:
    shapes_list = [(mapping(p), 1) for p in filtered_polys]
    final_pred_mask = rasterize(
        shapes_list,
        out_shape=(IMG_HEIGHT, IMG_WIDTH),
        transform=IMG_TRANSFORM,
        fill=0,
        dtype=np.uint8,
        all_touched=True
    ).astype(np.float32)
else:
    final_pred_mask = np.zeros((IMG_HEIGHT, IMG_WIDTH), dtype=np.float32)

# Compute final metrics
final_p, final_r, final_f1, final_iou = evaluate_at_threshold(final_pred_mask, mask_float, 0.5)

print('\n' + '=' * 40)
print('=== PHASE 1 RESULTS ===')
print('=' * 40)
print('Threshold sweep:')
for t in THRESHOLDS:
    r = sweep_results[t]
    marker = ' <-- best' if t == PRED_THRESHOLD else ''
    print(f'  {t:.2f}: P={r["precision"]:.4f} R={r["recall"]:.4f} F1={r["f1"]:.4f}{marker}')
print(f'Best threshold: {PRED_THRESHOLD:.2f}')
print()
print(f'Before geometric filter: {before_count:,} polygons')
print(f'After geometric filter:  {after_count:,} polygons (removed {removed:,})')
print()
print('FINAL METRICS (TTA + geom filter + best threshold):')
print(f'  Precision: {final_p:.4f}')
print(f'  Recall:    {final_r:.4f}')
print(f'  F1 Score:  {final_f1:.4f}')
print(f'  IoU:       {final_iou:.4f}')
print(f'  Polygons:  {after_count:,}')
print('=' * 40)

## 14. Save Outputs

In [ ]:
# Save eval.json
eval_data = {
    'phase': 1,
    'label_source': LABEL_SOURCE,
    'buildings_count': int(BUILDINGS_COUNT),
    'best_threshold': float(PRED_THRESHOLD),
    'threshold_sweep': {
        str(t): {k: float(v) for k, v in v2.items()}
        for t, v2 in sweep_results.items()
    },
    'before_geom_filter': int(before_count),
    'after_geom_filter':  int(after_count),
    'removed_by_filter':  int(removed),
    'final_metrics': {
        'precision': float(final_p),
        'recall':    float(final_r),
        'f1':        float(final_f1),
        'iou':       float(final_iou),
        'polygons':  int(after_count)
    },
    'training': {
        'epochs': NUM_EPOCHS,
        'best_epoch': int(best_epoch),
        'best_val_f1': float(best_val_f1),
        'batch_size': BATCH_SIZE,
        'chip_size': CHIP_SIZE,
        'stride_train': STRIDE_TRAIN,
        'tta_passes': 6
    },
    'geometry_filter': {
        'min_area_m2':  MIN_AREA_M2,
        'min_rect':     MIN_RECT,
        'min_compact':  MIN_COMPACT
    }
}

with open(str(EVAL_PATH), 'w') as f:
    json.dump(eval_data, f, indent=2)
print(f'eval.json saved: {EVAL_PATH}')

In [ ]:
# Save phase1_buildings.geojson
if filtered_polys:
    features = []
    for i, poly in enumerate(filtered_polys):
        area_m2 = poly.area * (111000 ** 2)
        features.append({
            'type': 'Feature',
            'geometry': mapping(poly),
            'properties': {
                'id':           i,
                'area_m2':      round(float(area_m2), 2),
                'rectangularity': round(float(rectangularity(poly)), 4),
                'compactness':    round(float(compactness(poly)), 4),
                'source':         'phase1_tta'
            }
        })
    geojson_out = {
        'type': 'FeatureCollection',
        'crs': {'type': 'name', 'properties': {'name': str(IMG_CRS)}},
        'features': features
    }
    with open(str(GEOJSON_PATH), 'w') as f:
        json.dump(geojson_out, f)
    print(f'GeoJSON saved: {GEOJSON_PATH} ({len(features):,} buildings)')
else:
    print('No polygons to save')

## 15. Visualization

In [ ]:
# 3-panel visualization: imagery | GT mask | TTA predictions
# Use a representative crop for visualization
VIZ_CROP_SIZE = min(512, IMG_HEIGHT, IMG_WIDTH)
cy = max(0, (IMG_HEIGHT - VIZ_CROP_SIZE) // 2)
cx = max(0, (IMG_WIDTH  - VIZ_CROP_SIZE) // 2)

img_crop  = IMG_ARRAY[cy:cy+VIZ_CROP_SIZE, cx:cx+VIZ_CROP_SIZE]
gt_crop   = MASK[cy:cy+VIZ_CROP_SIZE, cx:cx+VIZ_CROP_SIZE]
pred_crop = pred_mask[cy:cy+VIZ_CROP_SIZE, cx:cx+VIZ_CROP_SIZE]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(img_crop)
axes[0].set_title('RGB Imagery', fontsize=13)
axes[0].axis('off')

axes[1].imshow(img_crop)
axes[1].imshow(gt_crop, alpha=0.5, cmap='Reds', vmin=0, vmax=1)
axes[1].set_title(f'GT Mask — MS Buildings\n({BUILDINGS_COUNT:,} buildings)', fontsize=13)
axes[1].axis('off')

axes[2].imshow(img_crop)
axes[2].imshow(pred_crop > PRED_THRESHOLD, alpha=0.5, cmap='Blues', vmin=0, vmax=1)
axes[2].set_title(
    f'TTA Predictions (thr={PRED_THRESHOLD:.2f})\n'
    f'F1={final_f1:.4f} | P={final_p:.4f} | R={final_r:.4f}',
    fontsize=13
)
axes[2].axis('off')

red_patch  = mpatches.Patch(color='red',  alpha=0.6, label='Ground Truth')
blue_patch = mpatches.Patch(color='blue', alpha=0.6, label='Prediction')
fig.legend(handles=[red_patch, blue_patch], loc='lower center', ncol=2, fontsize=11)

plt.suptitle(
    'Predial Vision MX — Phase 1: TTA + Geometric Filter + Threshold Optimization\n'
    f'IoU={final_iou:.4f} | Polygons={after_count:,} | Label: {LABEL_SOURCE}',
    fontsize=12, fontweight='bold'
)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig(str(VIZ_PATH), dpi=150, bbox_inches='tight')
plt.show()
print(f'Visualization saved: {VIZ_PATH}')

## 16. Summary

In [ ]:
print('\n' + '=' * 50)
print('=== PHASE 1 RESULTS ===')
print('=' * 50)
print('Threshold sweep:')
for t in THRESHOLDS:
    r = sweep_results[t]
    marker = ' <-- best' if t == PRED_THRESHOLD else ''
    print(f'  {t:.2f}: P={r["precision"]:.4f} R={r["recall"]:.4f} F1={r["f1"]:.4f}{marker}')
print(f'Best threshold: {PRED_THRESHOLD:.2f}')
print()
print(f'Before geometric filter: {before_count:,} polygons')
print(f'After geometric filter:  {after_count:,} polygons (removed {removed:,})')
print()
print('FINAL METRICS (TTA + geom filter + best threshold):')
print(f'  Precision: {final_p:.4f}')
print(f'  Recall:    {final_r:.4f}')
print(f'  F1 Score:  {final_f1:.4f}')
print(f'  IoU:       {final_iou:.4f}')
print(f'  Polygons:  {after_count:,}')
print('=' * 50)
print()
print('Output files:')
for p in [MODEL_PATH, EVAL_PATH, GEOJSON_PATH, PROB_MAP_PATH, VIZ_PATH, CURVES_PATH]:
    exists = Path(p).exists()
    size   = Path(p).stat().st_size / 1e6 if exists else 0
    status = f'{size:.2f} MB' if exists else 'MISSING'
    print(f'  {Path(p).name:40s} {status}')